# Tax Document Classification Solution

## 1. Setup & Imports

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

from src.train_utils import load_training_data
from src.models import SemanticClassifier, HybridClassifier, SemanticOnlyClassifier
from src.pipeline import DocumentClassifierPipeline

## 2. Load and Prepare Data
Extract text from PDFs and align them with the target JSONs.

In [2]:
X_text, y_labels = load_training_data('../data', max_chars=200, include_background=True)
print(f"Total pages loaded: {len(X_text)}")

# Visualize class distribution
pd.Series(y_labels).value_counts()

Total pages loaded: 757


other      655
1040f       20
f1040s3     18
f1040sd     16
f1040s1     12
f8949       10
f1040sb      8
f1040sa      7
f1040se      5
f8889        4
f1040sc      2
Name: count, dtype: int64

## 3. Train Semantic Classifier
split the data (80/20) and train the Embeddings + Logistic Regression model.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X_text, y_labels, test_size=0.2, random_state=42, stratify=y_labels)

semantic_model = SemanticClassifier()
semantic_model.fit(X_train, y_train)

2026-02-05 06:33:16,188 - src.models - INFO - Encoding training data...
2026-02-05 06:33:16,189 - src.models - INFO - Loading Sentence Transformer model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

2026-02-05 06:33:22,049 - src.models - INFO - Training classifier...


,model_name,'all-MiniLM-L6-v2'


## 4. Evaluation

In [4]:
y_pred = semantic_model.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

              precision    recall  f1-score   support

       1040f       0.50      1.00      0.67         4
     f1040s1       0.40      1.00      0.57         2
     f1040s3       0.67      1.00      0.80         4
     f1040sa       1.00      1.00      1.00         1
     f1040sb       1.00      1.00      1.00         2
     f1040sd       1.00      1.00      1.00         3
     f1040se       1.00      1.00      1.00         1
       f8889       1.00      1.00      1.00         1
       f8949       1.00      1.00      1.00         2
       other       1.00      0.93      0.96       132

    accuracy                           0.94       152
   macro avg       0.86      0.99      0.90       152
weighted avg       0.97      0.94      0.95       152

Accuracy: 0.9408


In [ ]:
# Save weights
from pathlib import Path
model_path = Path("../models/semantic_classifier.pkl")  # use "../models/..." if cwd is notebooks/
model_path.parent.mkdir(parents=True, exist_ok=True)
semantic_model.save(str(model_path))
print(f"Saved to {model_path.resolve()}")

### Confusion Matrix

## 5. Hybrid Approach Verification
Test the Hybrid Classifier (Heuristic + Semantic) on the test set

In [ ]:
hybrid_model = HybridClassifier(semantic_classifier=semantic_model)

y_pred_hybrid = []
for text in X_test:
    res = hybrid_model.predict(text)
    y_pred_hybrid.append(res['label'])

print(classification_report(y_test, y_pred_hybrid))
print(f"Hybrid Accuracy: {accuracy_score(y_test, y_pred_hybrid):.4f}")

## 6. Full Pipeline Run
Run the pipeline on all input files and save results to `data/output`.

In [6]:
# Option 1: Hybrid Classifier (Heuristic + Semantic) -> data/output_hybrid
# hybrid_model wraps semantic_model with regex heuristics (create if not already from cell 11)
hybrid_model = HybridClassifier(semantic_classifier=semantic_model)
print("Running Hybrid Pipeline...")
pipeline_hybrid = DocumentClassifierPipeline(hybrid_model)
pipeline_hybrid.process_directory('../data/input', '../data/output_hybrid')

Running Hybrid Pipeline...
2026-02-05 06:37:19,391 - src.pipeline - INFO - Processing file: ../data/input/dummy4.pdf
2026-02-05 06:37:26,665 - src.pipeline - INFO - Inference: 1914.21 ms total (88 pages), 21.75 ms/page
2026-02-05 06:37:26,668 - src.pipeline - INFO - Saved result to ../data/output_hybrid/dummy4.json
2026-02-05 06:37:26,668 - src.pipeline - INFO - Processing file: ../data/input/dummy5.pdf
2026-02-05 06:37:32,664 - src.pipeline - INFO - Inference: 766.75 ms total (65 pages), 11.80 ms/page
2026-02-05 06:37:32,666 - src.pipeline - INFO - Saved result to ../data/output_hybrid/dummy5.json
2026-02-05 06:37:32,667 - src.pipeline - INFO - Processing file: ../data/input/dummy7.pdf
2026-02-05 06:37:38,112 - src.pipeline - INFO - Inference: 782.67 ms total (64 pages), 12.23 ms/page
2026-02-05 06:37:38,120 - src.pipeline - INFO - Saved result to ../data/output_hybrid/dummy7.json
2026-02-05 06:37:38,122 - src.pipeline - INFO - Processing file: ../data/input/dummy6.pdf
2026-02-05 06:3

In [7]:
# Option 2: Semantic Only Classifier -> data/output_semantic
print("\nRunning Semantic-Only Pipeline...")
# Ensure we use the wrapper with the updated default (200 chars)
semantic_only_wrapper = SemanticOnlyClassifier(semantic_model, max_chars=200)
pipeline_semantic = DocumentClassifierPipeline(semantic_only_wrapper)
pipeline_semantic.process_directory('../data/input', '../data/output_semantic')


Running Semantic-Only Pipeline...
2026-02-05 06:38:35,923 - src.pipeline - INFO - Processing file: ../data/input/dummy4.pdf
2026-02-05 06:38:43,432 - src.pipeline - INFO - Inference: 2660.50 ms total (88 pages), 30.23 ms/page
2026-02-05 06:38:43,442 - src.pipeline - INFO - Saved result to ../data/output_semantic/dummy4.json
2026-02-05 06:38:43,443 - src.pipeline - INFO - Processing file: ../data/input/dummy5.pdf
2026-02-05 06:38:50,518 - src.pipeline - INFO - Inference: 1462.93 ms total (65 pages), 22.51 ms/page
2026-02-05 06:38:50,526 - src.pipeline - INFO - Saved result to ../data/output_semantic/dummy5.json
2026-02-05 06:38:50,527 - src.pipeline - INFO - Processing file: ../data/input/dummy7.pdf
2026-02-05 06:38:56,490 - src.pipeline - INFO - Inference: 1357.97 ms total (64 pages), 21.22 ms/page
2026-02-05 06:38:56,493 - src.pipeline - INFO - Saved result to ../data/output_semantic/dummy7.json
2026-02-05 06:38:56,493 - src.pipeline - INFO - Processing file: ../data/input/dummy6.pdf